## Describing final layer neurons of ResNet-18 (Places)

In [1]:
import os
#virtually move to parent directory
os.chdir("..")

import torch
import pandas as pd

from sentence_transformers import SentenceTransformer

import clip
import utils
import data_utils
import similarity

/home/s4yadav/private/workspace/CLIP-dissect/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


In [2]:
#Arguments
clip_name = 'ViT-B/16'
target_name = 'resnet18_places'
target_layer = 'fc'
d_probe = 'broden'
concept_set = 'data/broden_labels_clean.txt'
batch_size = 4
device = 'cuda'
pool_mode = 'avg'

save_dir = 'saved_activations'
similarity_fn = similarity.soft_wpmi

In [3]:
utils.save_activations(clip_name = clip_name, target_name = target_name, target_layers = [target_layer], 
                       d_probe = d_probe, concept_set = concept_set, batch_size = batch_size, 
                       device = device, pool_mode=pool_mode, save_dir = save_dir)

/home/s4yadav/private/workspace/CLIP-dissect/data_utils.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('data/resnet18_places365.pth.tar')['stat

In [4]:
save_names = utils.get_save_names(clip_name = clip_name, target_name = target_name,
                                  target_layer = target_layer, d_probe = d_probe,
                                  concept_set = concept_set, pool_mode=pool_mode,
                                  save_dir = save_dir)

target_save_name, clip_save_name, text_save_name = save_names

similarities, target_feats = utils.get_similarity_from_activations(target_save_name, clip_save_name, 
                                                        text_save_name, similarity_fn, device=device)

with open(concept_set, 'r') as f: 
    words = (f.read()).split('\n')

/home/s4yadav/private/workspace/CLIP-dissect/utils.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  image_features = torch.load(clip_save_name, map_location='cpu').floa

torch.Size([365, 1197])


In [5]:
#Clean up names of target classes
with open('data/categories_places365.txt', 'r') as f:
    cls_id_to_name = f.read().split('\n')
    cls_id_to_name = [(cls[3:]).split(' ')[0] for cls in cls_id_to_name]

def process_word(word):
    if concept_set.startswith('data/broden_labels'):
        if word.endswith("-s"):
            word = word[:-2]
        word = word.replace('_', ' ')
        return "{}".format(word)
    elif concept_set == 'data/categories_places365.txt':
        
        word = word[3:].split(' ')[0]
        word = word.replace('/', '-')
        word = word.replace('_', ' ')
        
        return "{}".format(word)

# Accuracies

In [6]:
id_to_label = data_utils.get_places_id_to_broden_label()

def clean_label(label):
    if label.startswith('/'):
        label = label[3:]
        label = label.split(' ')[0]
    if label.endswith('-s'):
        label = label[:-2]
    return label
    
def get_topk_acc(similarities, k=5):
    correct = 0
    total = 0
    for orig_id in range(len(similarities)):
        #skip classes not in Broden
        if id_to_label[orig_id]==None:
            continue
        else:
            vals, ids = torch.topk(similarities[orig_id], k, largest=True)
            for idx in ids[:k]:
                if (process_word(words[idx])==process_word(id_to_label[orig_id])):
                    correct += 1
                    continue
            total += 1
    return (correct/total)*100

In [7]:
print("CLIP-Dissect Top 1 acc:{:.4f}".format(get_topk_acc(similarities, k=1)))
print("CLIP-Dissect Top 5 acc:{:.4f}".format(get_topk_acc(similarities, k=5)))

CLIP-Dissect Top 1 acc:58.8015
CLIP-Dissect Top 5 acc:86.1423


In [8]:
df = pd.read_csv('data/NetDissect_results/resnet18_places365_fc.csv')
correct = 0
total = 0
for i, label in enumerate(df['label']):
    if id_to_label[i]==None:
        continue
    else:
        correct += (clean_label(label)==clean_label(id_to_label[i]))
        total += 1

print("Network Dissection Top 1 acc:{:.4f}".format(correct/total*100))

Network Dissection Top 1 acc:43.8202


# Cos similarities

In [9]:
# Create a simple wrapper for sentence embeddings using transformers directly
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

class SimpleSentenceTransformer:
    def __init__(self, model_name):
        print(f"Loading {model_name} using transformers...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.model.eval()
    
    def encode(self, texts):
        # Convert numpy arrays and other iterables to list of strings
        if isinstance(texts, np.ndarray):
            texts = texts.tolist()
        elif not isinstance(texts, list):
            texts = list(texts)
        if isinstance(texts, str):
            texts = [texts]
        
        # Ensure all elements are strings
        texts = [str(t) for t in texts]
        
        encoded = self.tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
        encoded = {k: v.to(self.device) for k, v in encoded.items()}
        with torch.no_grad():
            output = self.model(**encoded)
            # Use mean pooling
            embeddings = output.last_hidden_state.mean(dim=1)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        return embeddings.cpu().numpy()

# Load models
try:
    model = SimpleSentenceTransformer('sentence-transformers/all-mpnet-base-v2')
except Exception as e:
    print(f"Failed with all-mpnet-base-v2: {e}")
    print("Using distilbert as fallback...")
    model = SimpleSentenceTransformer('distilbert-base-uncased')

clip_model, _ = clip.load(clip_name, device=device)

# model = SentenceTransformer('all-mpnet-base-v2')
# clip_model, _ = clip.load(clip_name, device=device)

Loading sentence-transformers/all-mpnet-base-v2 using transformers...


In [10]:
clip_preds = torch.argmax(similarities, dim=1)
clip_preds = [words[int(pred)] for pred in clip_preds]

clip_cos, mpnet_cos = utils.get_cos_similarity(clip_preds, cls_id_to_name, clip_model, model, device, batch_size)
print("CLIP-Dissect - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

CLIP-Dissect - Clip similarity: 0.9116, mpnet similarity: 0.7203


In [11]:
netdissect_res = pd.read_csv('data/NetDissect_results/resnet18_places365_fc.csv')
nd_preds = netdissect_res['label'].values
nd_preds = [clean_label(pred) for pred in nd_preds]

clip_cos, mpnet_cos = utils.get_cos_similarity(nd_preds, cls_id_to_name, clip_model, model, device, batch_size)
print("Network Dissection - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

Network Dissection - Clip similarity: 0.8887, mpnet similarity: 0.7028
